## Imports

In [20]:
import cv2
import numpy as np
import os
import sys
import time
import logging
from datetime import datetime

## Configuration Step

In [21]:

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('object_detection.log'),
        logging.StreamHandler(sys.stdout)
    ]
)

# Configuration - Easy to modify for deployment
CONFIG = {
    "prototxt_path": r"C:\Users\HP\Documents\Real-Time-Object-detection\deploy.prototxt.txt",
    "model_path": r"C:\Users\HP\Documents\Real-Time-Object-detection\mobilenet_iter_73000.caffemodel",
    "max_frames": 300,                # Set to 0 for continuous operation
    "confidence_threshold": 0.5,
    "output_dir": "detection_results",
    "display_output": True,            # Set to False for headless systems
    "save_video": True,
    "log_interval": 10,                # Log every N frames
    "classes": [
        "background", "aeroplane", "bicycle", "bird", "boat",
        "bottle", "bus", "car", "cat", "chair", "cow", "diningtable",
        "dog", "horse", "motorbike", "person", "pottedplant", "sheep",
        "sofa", "train", "tvmonitor"
    ]
}

## Define the Object Detector Class

In [22]:

class ObjectDetector:
    def __init__(self, config):
        self.config = config
        self.validate_paths()
        self.net = self.load_model()
        self.cap = self.initialize_camera()
        self.output_writer = None
        self.frame_count = 0
        self.start_time = time.time()
        self.object_log = []
        
        # Create output directory
        os.makedirs(config["output_dir"], exist_ok=True)
        
    def validate_paths(self):
        if not os.path.isfile(self.config["prototxt_path"]):
            raise FileNotFoundError(f"Prototxt file not found at {self.config['prototxt_path']}")
        if not os.path.isfile(self.config["model_path"]):
            raise FileNotFoundError(f"Model file not found at {self.config['model_path']}")

    def load_model(self):
        logging.info("Loading detection model...")
        return cv2.dnn.readNetFromCaffe(
            self.config["prototxt_path"], 
            self.config["model_path"]
        )

    def initialize_camera(self):
        logging.info("Initializing video source...")
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            raise RuntimeError("Could not open video source")
        return cap

    def setup_output_video(self):
        frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = self.cap.get(cv2.CAP_PROP_FPS) or 20.0
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(
            self.config["output_dir"],
            f"detection_{timestamp}.avi"
        )
        
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        self.output_writer = cv2.VideoWriter(
            output_path, fourcc, fps, (frame_width, frame_height)
        )
        logging.info(f"Output video will be saved to: {output_path}")
        return output_path

    def process_frame(self, frame):
        self.frame_count += 1
        object_counts = {}
        
        # Perform detection
        blob = cv2.dnn.blobFromImage(
            cv2.resize(frame, (300, 300)),
            0.007843, 
            (300, 300), 
            127.5
        )
        self.net.setInput(blob)
        detections = self.net.forward()

        # Process detections
        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > self.config["confidence_threshold"]:
                idx = int(detections[0, 0, i, 1])
                if idx >= len(self.config["classes"]):
                    continue
                    
                label = self.config["classes"][idx]
                object_counts[label] = object_counts.get(label, 0) + 1

                # Extract and validate bounding box
                box = detections[0, 0, i, 3:7] * np.array([
                    frame.shape[1], frame.shape[0], 
                    frame.shape[1], frame.shape[0]
                ])
                startX, startY, endX, endY = box.astype("int")
                startX = max(0, startX); startY = max(0, startY)
                endX = min(frame.shape[1]-1, endX)
                endY = min(frame.shape[0]-1, endY)
                
                if endX > startX and endY > startY:
                    color = self.get_color(label)
                    cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
                    text = f"{label}: {confidence*100:.1f}%"
                    y = startY - 10 if startY > 20 else startY + 20
                    cv2.putText(frame, text, (startX, y), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
        
        # Log detection results periodically
        if self.frame_count % self.config["log_interval"] == 0:
            self.log_detections(object_counts)
        
        # Add frame metadata to log
        self.object_log.append({
            "frame": self.frame_count,
            "timestamp": time.time(),
            "objects": object_counts
        })
        
        # Add info overlay
        self.add_info_overlay(frame, object_counts)
        
        return frame, object_counts

    def get_color(self, label):
        # Simple color coding for common objects
        color_map = {
            "person": (0, 0, 255),    # Red
            "car": (0, 255, 255),     # Yellow
            "dog": (255, 0, 0),       # Blue
            "cat": (255, 0, 255),     # Magenta
        }
        return color_map.get(label, (0, 255, 0))  # Green for others

    def add_info_overlay(self, frame, object_counts):
        # Frame counter and FPS
        elapsed = time.time() - self.start_time
        fps = self.frame_count / elapsed if elapsed > 0 else 0
        status = f"Frame: {self.frame_count} | FPS: {fps:.1f}"
        cv2.putText(frame, status, (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Object counts
        y_offset = 60
        for obj, count in object_counts.items():
            cv2.putText(frame, f"{obj}: {count}", (10, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            y_offset += 30

    def log_detections(self, object_counts):
        logging.info(f"Frame {self.frame_count} - Detected objects: {object_counts}")

    def run(self):
        logging.info("Starting object detection system")
        
        if self.config["save_video"]:
            video_path = self.setup_output_video()
        
        try:
            while True:
                ret, frame = self.cap.read()
                if not ret:
                    logging.warning("Frame capture error, skipping...")
                    continue
                
                processed_frame, _ = self.process_frame(frame)
                
                # Save to video
                if self.config["save_video"] and self.output_writer:
                    self.output_writer.write(processed_frame)
                
                # Display output
                if self.config["display_output"]:
                    cv2.imshow("Object Detection", processed_frame)
                    cv2.waitKey(1)  # Minimal delay
                
                # Check frame limit
                if self.config["max_frames"] > 0 and self.frame_count >= self.config["max_frames"]:
                    logging.info(f"Reached max frames limit ({self.config['max_frames']})")
                    break
        
        except KeyboardInterrupt:
            logging.info("Interrupted by user")
        except Exception as e:
            logging.error(f"Processing error: {str(e)}")
        finally:
            self.cleanup()
            self.generate_report(video_path if self.config["save_video"] else None)
    
    def cleanup(self):
        logging.info("Cleaning up resources...")
        self.cap.release()
        if self.output_writer:
            self.output_writer.release()
        if self.config["display_output"]:
            cv2.destroyAllWindows()
    
    def generate_report(self, video_path):
        logging.info("Generating detection report...")
        
        # Calculate statistics
        elapsed = time.time() - self.start_time
        fps = self.frame_count / elapsed if elapsed > 0 else 0
        
        # Create summary file
        report_path = os.path.join(self.config["output_dir"], "detection_summary.txt")
        with open(report_path, 'w') as f:
            f.write(f"Object Detection Report\n")
            f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total frames processed: {self.frame_count}\n")
            f.write(f"Processing time: {elapsed:.2f} seconds\n")
            f.write(f"Average FPS: {fps:.1f}\n")
            
            if video_path:
                f.write(f"\nOutput video: {video_path}\n")
            
            f.write("\nObject Detection Summary:\n")
            
            # Calculate total object counts
            total_counts = {}
            for entry in self.object_log:
                for obj, count in entry["objects"].items():
                    total_counts[obj] = total_counts.get(obj, 0) + count
            
            for obj, count in sorted(total_counts.items()):
                f.write(f"{obj}: {count} detections\n")
        
        logging.info(f"Report saved to {report_path}")
        logging.info("Processing complete")



## Run the Detector

In [23]:
if __name__ == "__main__":
    detector = ObjectDetector(CONFIG)
    detector.run()

2025-06-23 22:34:45,254 - INFO - Loading detection model...
2025-06-23 22:34:45,327 - INFO - Initializing video source...
2025-06-23 22:34:47,649 - INFO - Starting object detection system
2025-06-23 22:34:47,652 - INFO - Output video will be saved to: detection_results\detection_20250623_223447.avi
2025-06-23 22:34:48,935 - INFO - Frame 10 - Detected objects: {'bottle': 1}
2025-06-23 22:34:49,418 - INFO - Frame 20 - Detected objects: {'bottle': 1}
2025-06-23 22:34:49,926 - INFO - Frame 30 - Detected objects: {'bottle': 1}
2025-06-23 22:34:50,398 - INFO - Frame 40 - Detected objects: {'bottle': 1}
2025-06-23 22:34:50,859 - INFO - Frame 50 - Detected objects: {'bottle': 1}
2025-06-23 22:34:51,331 - INFO - Frame 60 - Detected objects: {'bottle': 1}
2025-06-23 22:34:51,822 - INFO - Frame 70 - Detected objects: {'bottle': 1}
2025-06-23 22:34:52,300 - INFO - Frame 80 - Detected objects: {'bottle': 1}
2025-06-23 22:34:52,752 - INFO - Frame 90 - Detected objects: {'bottle': 1}
2025-06-23 22:34